| Model | Type | Dataset 1 (Accuracy/F1/AUROC) | Dataset 2 (Accuracy/F1/AUROC) |
|-------|------|------------------------------|------------------------------|
| [BERT](https://github.com/google-research/bert) | Transformer | 90.2% / 0.85 / 0.76 | 88.1% / 0.82 / 0.73 |
| [RoBERTa](https://github.com/facebookresearch/fairseq/tree/main/examples/roberta) | Transformer | 92.5% / 0.89 / 0.81 | 91.0% / 0.87 / 0.79 |
| [Our Model](link/to/your/repo) | Custom | **94.1%** / **0.91** / **0.85** | **93.2%** / **0.90** / **0.84** |

<table>
  <tr>
    <th>Model</th>
    <th colspan="3" align="center">Dataset 1</th>
    <th colspan="3" align="center">Dataset 2</th>
  </tr>
  <tr>
    <th></th>
    <th>Accuracy</th>
    <th>F1-Score</th>
    <th>AUROC</th>
    <th>Accuracy</th>
    <th>F1-Score</th>
    <th>AUROC</th>
  </tr>
  <tr>
    <td>Model 1</td>
    <td>90.2%</td>
    <td>0.85</td>
    <td>0.76</td>
    <td>88.1%</td>
    <td>0.82</td>
    <td>0.73</td>
  </tr>
  <tr>
    <td>Model 2</td>
    <td>92.5%</td>
    <td>0.89</td>
    <td>0.81</td>
    <td>91.0%</td>
    <td>0.87</td>
    <td>0.79</td>
  </tr>
  <tr>
    <td>Model 3</td>
    <td><b>94.1%</b></td>
    <td><b>0.91</b></td>
    <td><b>0.85</b></td>
    <td><b>93.2%</b></td>
    <td><b>0.90</b></td>
    <td><b>0.84</b></td>
  </tr>
</table>

In [1]:
import tsl
import torch
import numpy as np
import pandas as pd
from tsl.datasets import MetrLA
from einops import rearrange
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.nn import models
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn

import random
import torch
import numpy as np
import os

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler
from pytorch_lightning.profilers import AdvancedProfiler

from torch.optim.lr_scheduler import MultiStepLR
from pytorch_lightning.loggers import TensorBoardLogger



def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision('medium')  # 'medium' favors performance over precision

        # Enable TF32 format which is optimized for Tensor Cores on Ampere+ GPUs
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

seed_everything(42)

42

In [2]:
dataset = MetrLA(root='./data/metrla')

connectivity = dataset.get_connectivity(threshold=0.1,
                                        include_self=False,
                                        normalize_axis=1,
                                        force_symmetric=True,
                                        layout="edge_index")

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = SpatioTemporalDataset(target=dataset.dataframe(),
                                      connectivity=connectivity,
                                      mask=dataset.mask,
                                      covariates=covariates,
                                      horizon=12,
                                      window=12,
                                      stride=1)
print(torch_dataset)

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:98: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(df.index[0], df.index[-1], freq='5T')
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:109: FutureWarning: The 'method' keyword in DataFrame.replace is deprecated and will be removed in a future version.
  df = df.replace(to_replace=0., method='ffill')


SpatioTemporalDataset(n_samples=34249, n_nodes=207, n_channels=1)


In [3]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=64,
    workers = 4
)

dm.setup()
print(dm)

{Train dataloader: size=24648}
{Validation dataloader: size=2728}
{Test dataloader: size=6849}
{Predict dataloader: None}


In [4]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
    'mae': torch_metrics.MaskedMAE(),
    'mse': torch_metrics.MaskedMSE(),
    'mae_step_1': torch_metrics.MaskedMAE(at=0),
   'mae_step_2': torch_metrics.MaskedMAE(at=2),
   'mae_step_3': torch_metrics.MaskedMAE(at=5),
   'mae_step_4': torch_metrics.MaskedMAE(at=11)
}

model = models.GraphWaveNetModel(input_size=1,exog_size=2, hidden_size = 64, output_size=1,spatial_kernel_size=4,
                         horizon=12, ff_size = 32, dropout = 0.1,n_layers = 8,n_nodes=torch_dataset.n_nodes)

def get_model_log_name(model, torch_dataset):
    class_name = model.__class__.__name__
    directed = str(not is_undirected(torch_dataset.edge_index))
    return f"{class_name}_directed_{directed}"
    

logger = TensorBoardLogger(
        save_dir=f"logs/{dataset.name}",
        name=get_model_log_name(model,torch_dataset)
)

In [5]:
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  'weight_decay':1e-3
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[40, 80, 120]}
)
# 'momentum':0.9,
#                  'nesterov':True

In [6]:
checkpoint_callback = ModelCheckpoint(
    dirpath=f'model_checkpoint/{dataset.name}/{model.__class__.__name__}',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
    verbose=True,
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=30,
        mode='min'
    )

trainer = Trainer(
        max_epochs=200,
        limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=[1],
        gradient_clip_val=5,
       callbacks=[checkpoint_callback, early_stop_callback],
      # default_root_dir="logs",
        # profiler=profiler,
        precision = '16-mixed',
        check_val_every_n_epoch = 5,
    logger=logger

    
)

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [7]:
trainer.fit(predictor, datamodule=dm)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type              | Params | Mode 
------------------------------------------------------------
0 | loss_fn       | MaskedMAE         | 0      | train
1 | train_metrics | MetricCollection  | 0      | train
2 | val_metrics   | MetricCollection  | 0      | train
3 | test_metrics  | MetricCollection  | 0      | train
4 | model         | GraphWaveNetModel | 584 K  | train
------------------------------------------------------------
584 K     Trainable params
0         Non-trainable params
584 K     Total params
2.336     Total estimated model params size (MB)
171       Modules in train mode
0         Modules in eval mode


Training: |                                                                                                   …

Only args ['u', 'edge_weight', 'x', 'edge_index'] are forwarded to the model (GraphWaveNetModel).


Validation: |                                                              | 0/? [00:00<?, ?it/s]

Epoch 4, global step 750: 'val_mae' reached 3.04936 (best 3.04936), saving model to '/netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MetrLA/GraphWaveNetModel/epoch=4-step=750-v1.ckpt' as top 1


...

Epoch 189, global step 28500: 'val_mae' was not in top 1


Validation: |                                                                                                 …

Epoch 194, global step 29250: 'val_mae' was not in top 1


Validation: |                                                                                                 …

Epoch 199, global step 30000: 'val_mae' was not in top 1
`Trainer.fit` stopped: `max_epochs=200` reached.


In [8]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MetrLA/GraphWaveNetModel/epoch=169-step=25500.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /netfs/tsp/student/2022/zhu/ST_RUM/model_checkpoint/MetrLA/GraphWaveNetModel/epoch=169-step=25500.ckpt


Testing: |                                                                                                    …

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │    3.1203155517578125     │
│         test_mae          │     3.235675811767578     │
│      test_mae_step_1      │     2.327878713607788     │
│      test_mae_step_2      │     2.836237907409668     │
│      test_mae_step_3      │     3.144770860671997     │
│      test_mae_step_4      │     3.379467248916626     │
│         test_mse          │    41.089359283447266     │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 3.235675811767578,
  'test_mae_step_1': 2.327878713607788,
  'test_mae_step_2': 2.836237907409668,
  'test_mae_step_3': 3.144770860671997,
  'test_mae_step_4': 3.379467248916626,
  'test_mse': 41.089359283447266,
  'test_loss': 3.1203155517578125}]